In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

device = torch.device("cpu")
device

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
batch_size = 256

tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_ds = datasets.FashionMNIST(root="./data", train=True, download=True, transform=tfm)
test_ds = datasets.FashionMNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

classes = train_ds.classes
len(train_ds), len(test_ds), classes

In [ ]:
def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> dict:
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1
    return {
        "loss": total_loss / max(1, n_batches),
        "acc": total_acc / max(1, n_batches)
    }


def train_epochs(model: nn.Module, train_loader: DataLoader, test_loader: DataLoader, *,
                 epochs: int, lr: float, weight_decay: float) -> dict:
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
    }

    for _ in tqdm(range(epochs)):
        model.train()
        epoch_loss = 0.0
        epoch_acc = 0.0
        n_batches = 0
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            epoch_acc += accuracy_from_logits(logits, y)
            n_batches += 1

        train_metrics = {
            "loss": epoch_loss / max(1, n_batches),
            "acc": epoch_acc / max(1, n_batches)
        }
        test_metrics = evaluate(model, test_loader, criterion)

        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["acc"])
        history["test_loss"].append(test_metrics["loss"])
        history["test_acc"].append(test_metrics["acc"])

    return history


def plot_history(h: dict, title: str):
    epochs = np.arange(1, len(h["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].plot(epochs, h["train_loss"], label="train")
    ax[0].plot(epochs, h["test_loss"], label="test")
    ax[0].set_title(f"Loss — {title}")
    ax[0].set_xlabel("epoch")
    ax[0].set_ylabel("loss")
    ax[0].grid(True)
    ax[0].legend()

    ax[1].plot(epochs, h["train_acc"], label="train")
    ax[1].plot(epochs, h["test_acc"], label="test")
    ax[1].set_title(f"Accuracy — {title}")
    ax[1].set_xlabel("epoch")
    ax[1].set_ylabel("acc")
    ax[1].grid(True)
    ax[1].legend()

    plt.show()


def compare_histories(histories: dict, key: str, title: str):
    fig, ax = plt.subplots(figsize=(8, 4))
    for name, h in histories.items():
        epochs = np.arange(1, len(h[key]) + 1)
        ax.plot(epochs, h[key], label=name)
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.grid(True)
    ax.legend()
    plt.show()

In [ ]:
class MLP(nn.Module):
    def __init__(self, activation: str, use_bn: bool, dropout_p: float):
        super().__init__()
        act = {
            "relu": nn.ReLU(),
            "sigmoid": nn.Sigmoid(),
        }[activation]

        layers = []
        in_dim = 28 * 28
        hidden = [512, 256, 128]
        for h in hidden:
            layers.append(nn.Linear(in_dim, h))
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(act)
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            in_dim = h
        layers.append(nn.Linear(in_dim, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)


mlp_reg = MLP(activation="relu", use_bn=True, dropout_p=0.3)
h_reg = train_epochs(mlp_reg, train_loader, test_loader, epochs=15, lr=1e-3, weight_decay=1e-4)
plot_history(h_reg, "MLP ReLU + regularization")
h_reg["test_acc"][-1], h_reg["test_loss"][-1]

In [ ]:
mlp_noreg = MLP(activation="relu", use_bn=False, dropout_p=0.0)
h_noreg = train_epochs(mlp_noreg, train_loader, test_loader, epochs=15, lr=1e-3, weight_decay=0.0)
plot_history(h_noreg, "MLP ReLU (no regularization)")

compare_histories({
    "reg test_loss": {"test_loss": h_reg["test_loss"]},
    "no reg test_loss": {"test_loss": h_noreg["test_loss"]},
}, key="test_loss", title="Test loss сравнение")

compare_histories({
    "reg test_acc": {"test_acc": h_reg["test_acc"]},
    "no reg test_acc": {"test_acc": h_noreg["test_acc"]},
}, key="test_acc", title="Test accuracy сравнение")

h_noreg["test_acc"][-1], h_noreg["test_loss"][-1]

In [ ]:
mlp_sig = MLP(activation="sigmoid", use_bn=True, dropout_p=0.3)
h_sig = train_epochs(mlp_sig, train_loader, test_loader, epochs=15, lr=1e-3, weight_decay=1e-4)
plot_history(h_sig, "MLP Sigmoid + regularization")

compare_histories({
    "ReLU reg test_acc": {"test_acc": h_reg["test_acc"]},
    "Sigmoid reg test_acc": {"test_acc": h_sig["test_acc"]},
}, key="test_acc", title="ReLU vs Sigmoid (test accuracy)")

compare_histories({
    "ReLU reg test_loss": {"test_loss": h_reg["test_loss"]},
    "Sigmoid reg test_loss": {"test_loss": h_sig["test_loss"]},
}, key="test_loss", title="ReLU vs Sigmoid (test loss)")

h_sig["test_acc"][-1], h_sig["test_loss"][-1]

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 3 * 3, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


cnn = SimpleCNN()
h_cnn = train_epochs(cnn, train_loader, test_loader, epochs=10, lr=1e-3, weight_decay=1e-4)
plot_history(h_cnn, "Simple CNN")
h_cnn["test_acc"][-1], h_cnn["test_loss"][-1]

In [ ]:
@torch.no_grad()
def get_activations_first_layers(model: SimpleCNN, x: torch.Tensor):
    a1 = F.relu(model.conv1(x))
    p1 = model.pool(a1)
    a2 = F.relu(model.conv2(p1))
    return a1, a2


x_batch, y_batch = next(iter(test_loader))
x = x_batch[:2].to(device)

cnn = cnn.to(device)
cnn.eval()
a1, a2 = get_activations_first_layers(cnn, x)
a1 = a1.cpu()
a2 = a2.cpu()

def show_feature_maps(img: torch.Tensor, maps: torch.Tensor, n_maps: int, title: str):
    n_maps = min(n_maps, maps.size(0))
    cols = 8
    rows = math.ceil(n_maps / cols)
    fig = plt.figure(figsize=(2 * cols, 2 * (rows + 1)))

    ax0 = fig.add_subplot(rows + 1, cols, 1)
    ax0.imshow(img.squeeze(0), cmap="gray")
    ax0.set_title("input")
    ax0.axis("off")

    for i in range(1, cols):
        ax = fig.add_subplot(rows + 1, cols, 1 + i)
        ax.axis("off")

    for i in range(n_maps):
        r = (i // cols) + 1
        c = (i % cols)
        ax = fig.add_subplot(rows + 1, cols, r * cols + c + 1)
        ax.imshow(maps[i], cmap="viridis")
        ax.set_title(str(i))
        ax.axis("off")

    plt.suptitle(title)
    plt.show()


for idx in range(x.size(0)):
    show_feature_maps(x[idx].cpu(), a1[idx], n_maps=16, title=f"Conv1 activations (sample {idx})")
    show_feature_maps(x[idx].cpu(), a2[idx], n_maps=16, title=f"Conv2 activations (sample {idx})")